In [48]:
import importlib, tempfile, collections
import pandas as pd
import scoring, dsh, stage3
importlib.reload(stage3)  
import json

import table_solver as tab, fit_solver as fit, calc_solver as calc
for m in (tab, fit, calc):
    importlib.reload(m)

In [29]:
dev = json.load(open("dev.json", encoding='utf-8'))

Check for how many exercises no llm was used and if it was done correctly

In [33]:
def solve_one(t):
    if not t['files']:
        return calc.calc_general(t)
    p = scoring.write_files([t], tempfile.mkdtemp())[0]          
    if t['type'].startswith('fit'):
        return fit.solve_fit_task(p)
    try:
        return dsh.to_number(tab.solve_data(p))
    except Exception:
        return dsh.to_number(tab.solve_table_semantic(p))

rows = []
for t in dev:
    try:
        a = solve_one(t)
        status, reason = ('right' if scoring.score_answer(t, a)[0] == 1 else 'WRONG NUMBER'), '' # right answer without using llm
    except fit.Unclear as e:
        a, status, reason = None, 'unclear (fit used LLM)', str(e.phrases) # llm was used for fit (not shown if correctly answered)
    except Exception as e:
        a, status, reason = None, 'open (LLM)', f'{type(e).__name__}: {e}' # llm was used for table and calc (not shown if correctly answered)
    rows.append((t['id'], t['type'], status, a, t['answer'], reason))

res = pd.DataFrame(rows, columns=['id', 'type', 'status', 'answer', 'gold', 'reason'])
pd.set_option('display.width', 250)
print(pd.crosstab(res.type, res.status).reindex(columns=['right', 'WRONG NUMBER', 'unclear (fit used LLM)', 'open (LLM)'], fill_value=0).assign(All=lambda x: x.sum(axis=1)))           

status             right  WRONG NUMBER  unclear (fit used LLM)  open (LLM)  All
type                                                                           
calc.arithmetic        8             0                       0           0    8
calc.dates             8             0                       0           0    8
calc.functions         7             0                       0           0    7
calc.growth            7             0                       0           0    7
calc.integer           7             0                       0           0    7
calc.percent           7             0                       0           0    7
calc.probability       7             0                       0           0    7
calc.stats_inline      7             0                       0           0    7
calc.word              7             0                       0           0    7
fit.linear            18             0                       0           0   18
fit.logistic          17             0  

Get all those that were incorrect

In [34]:
pd.set_option('display.max_colwidth', 150)
print(res[res.status != 'right'].sort_values(['status', 'type']).to_string())        

Empty DataFrame
Columns: [id, type, status, answer, gold, reason]
Index: []


In [31]:
# code to get the error messages
import traceback

rows = []
for t in dev:
    try:
        a = solve_one(t)
        status, reason = (
            'right' if scoring.score_answer(t, a)[0] == 1 else 'WRONG NUMBER',
            ''
        )
    except fit.Unclear as e:
        a, status, reason = None, 'unclear (fit used LLM)', str(e.phrases)
    except Exception as e:
        traceback.print_exc()  # zeigt die genaue Fehlerzeile
        a, status, reason = None, 'open (LLM)', f'{type(e).__name__}: {e}'

    rows.append((t['id'], t['type'], status, a, t['answer'], reason))

In [5]:
# choose a certain type if too many errrors are shown before
res[res.type.str.startswith('table')].reason.str[:120].value_counts().head(10)

reason
    80
Name: count, dtype: int64

In [ ]:
# if possible, load agent to check agent
importlib.reload(stage3)
agent = stage3.Agent()  

ids = list(res[res.status == 'unclear (fit used LLM)'].id)
batch = [t for t in dev if t['id'] in ids]
out = agent.solve(scoring.write_files(batch, tempfile.mkdtemp()), seconds=120)
for t, o in zip(batch, out):
    print(t['id'], '| Answer:', o['answer'], '| gold:', t['answer'],
          '| right' if scoring.score_answer(t, o['answer'])[0] == 1 else '| wrong')

NameError: name 'agent' is not defined

--- not important ---

Problem that was solved but to keep it in mind:

In [23]:
# problem r2 is read as RÂ²
t = next(t for t in dev if t['id'] == 'd_012')
print('RÂ²' in t['objective'], 'R²' in t['objective'])

True False


check the new data

In [38]:
import os, sys

In [42]:
def check(tasks, show=0, only=None, solver_dir='.'):
    path = os.path.abspath(solver_dir)
    if path not in sys.path:
        sys.path.insert(0, path)
    mods = [importlib.reload(sys.modules[m]) if m in sys.modules else importlib.import_module(m)
            for m in ('scoring', 'fit_solver', 'calc_solver', 'table_solver')]
    scoring, fit, calc, tab = mods
    res, wrong = collections.defaultdict(collections.Counter), collections.defaultdict(list)
    for t in tasks:
        if only and not t['type'].startswith(only):
            continue
        fam = t['type'].split('.')[0]; p = scoring.write_files([t], tempfile.mkdtemp())[0]
        try:
            a = calc.calc_general(p) if fam == 'calc' else fit.solve_fit_task(p) if fam == 'fit' else tab.solve_data(p)
            s = 'right' if scoring.score_answer(t, float(a))[0] == 1 else 'WRONG'
            if s == 'WRONG': wrong[t['type']].append((float(a), t['answer'], t['objective']))
        except Exception:
            s = 'fallback'
        res[t['type']][s] += 1
    for ty, lst in wrong.items():
        for a, g, o in lst[:show]: print(f'[{ty}] answer {a:.6g}, gold {g}\n  {o}\n')
    table = pd.DataFrame(res).T.reindex(columns=['right', 'WRONG', 'fallback']).fillna(0).astype(int).sort_index()
    table.loc['TOTAL'] = table.sum()
    return table

In [49]:
def load(path='gen.json'):
    return json.load(open(path, encoding='utf-8'))

tasks = load('other_data.json')
check(tasks, show = 1)

[table.percentile] answer 67, gold 112.0
  You are given daily electricity readings of a company's sites in meter.csv, with a data dictionary. What is the 75th percentile of the people present over all the readings (in people)? meter.csv uses ',' to separate columns and a point as the decimal mark. Missing values are written '--'. Ignore values that are missing or not recorded. meter.csv contains duplicated rows (same identifier); keep one copy of each. Give the answer rounded to two decimal places.



,right,WRONG,fallback
calc.arithmetic,49,0,11
calc.dates,57,0,17
calc.functions,53,0,20
calc.growth,54,0,7
calc.integer,43,0,24
calc.percent,32,0,35
calc.probability,35,0,25
calc.stats_inline,47,0,3
calc.word,29,0,34
fit.linear,151,0,0
